# Notebook 3: Two-Stage Stochastic Programming for BRP under Uncertainty

## References

1. **Morales, J. M., Conejo, A. J., Pérez-Ruiz, J. (2010).** *"Short-term trading for a wind power producer."* IEEE Transactions on Power Systems, 25(1), 554–564. [DOI: 10.1109/TPWRS.2009.2036810](https://doi.org/10.1109/TPWRS.2009.2036810)

2. **Conejo, A. J., Carrión, M., Morales, J. M. (2010).** *Decision Making Under Uncertainty in Electricity Markets.* Springer. [DOI: 10.1007/978-1-4419-7421-1](https://doi.org/10.1007/978-1-4419-7421-1)

3. **Boomsma, T. K., Juul, N., Fleten, S.-E. (2014).** *"Bidding in sequential electricity markets: The Nordic case."* European Journal of Operational Research, 238(3), 797–809. [DOI: 10.1016/j.ejor.2014.04.027](https://doi.org/10.1016/j.ejor.2014.04.027)

4. **Pflug, G. Ch., Pichler, A. (2014).** *Multistage Stochastic Optimization.* Springer. [DOI: 10.1007/978-3-319-08843-3](https://doi.org/10.1007/978-3-319-08843-3)

## What these papers bring

**Morales et al. (2010)** introduce a two-stage stochastic programming framework for a wind power producer participating in a day-ahead market. The first stage determines the optimal day-ahead offer (nomination), while the second stage models real-time balancing under multiple scenarios of wind realization. This is directly analogous to our BRP problem.

**Conejo et al. (2010)** provide the foundational textbook treatment of decision-making under uncertainty in electricity markets, including extensive-form and scenario-based stochastic programming methods.

**Boomsma et al. (2014)** extend the framework to sequential markets (day-ahead → intraday → balancing), showing that the value of flexibility increases substantially under uncertainty.

**Pflug & Pichler (2014)** offer rigorous mathematical foundations for multistage stochastic optimization with applications in energy.

## What is implemented below

We implement a **two-stage stochastic MILP** for a BRP with multiple prosumer sites:
- **Stage 1 (here-and-now)**: Day-ahead nomination decisions (before uncertainty is revealed)
- **Stage 2 (recourse)**: Real-time BESS dispatch under each scenario

We generate scenarios for PV and load forecast errors, then solve the problem to find the nomination that minimizes **expected total cost** (DA cost + expected imbalance cost).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pulp

np.random.seed(42)


## 1. Scenario Generation

We generate $S$ scenarios for PV generation and load consumption deviations from forecast. Each scenario represents a possible realization of the uncertain quantities. In practice, these scenarios would come from ensemble weather forecasts or historical error distributions.


In [ ]:
T = 24
hours = np.arange(T)
N_sites = 3  # Reduced for tractability
S = 10       # Number of scenarios

# Site parameters
site_params = [
    (6.0, 2.0, 10.0, 5.0),
    (8.0, 3.0, 12.0, 6.0),
    (4.0, 1.5, 7.0, 3.5),
]

# Forecast (expected) profiles
pv_forecast = []
load_forecast = []
for i, (pv_peak, load_base, _, _) in enumerate(site_params):
    pv_f = np.maximum(0, pv_peak * np.exp(-0.5 * ((hours - 12) / 3.0)**2))
    pv_forecast.append(pv_f)
    load_f = load_base + 0.8 * np.exp(-0.5 * ((hours - 7.5) / 2.0)**2) + \
             1.2 * np.exp(-0.5 * ((hours - 19) / 2.5)**2)
    load_forecast.append(load_f)

# Generate scenarios: multiplicative perturbation of forecasts
# PV has higher uncertainty (cloud cover), load has moderate uncertainty
pv_scenarios = []   # shape: [scenario][site][time]
load_scenarios = []

scenario_probs = np.ones(S) / S  # Equal probability

for s in range(S):
    pv_s = []
    load_s = []
    for i in range(N_sites):
        # PV: multiplicative noise with temporal correlation
        pv_noise = 1.0 + 0.3 * np.random.randn(T)
        pv_noise = np.convolve(pv_noise, np.ones(3)/3, mode='same')  # smooth
        pv_realization = np.maximum(0, pv_forecast[i] * pv_noise)
        pv_s.append(pv_realization)

        # Load: smaller noise
        load_noise = 1.0 + 0.1 * np.random.randn(T)
        load_noise = np.convolve(load_noise, np.ones(3)/3, mode='same')
        load_realization = np.maximum(0.3, load_forecast[i] * load_noise)
        load_s.append(load_realization)

    pv_scenarios.append(pv_s)
    load_scenarios.append(load_s)

# Day-ahead prices
price_da = 40 + 20 * np.sin(2 * np.pi * (hours - 6) / 24) + \
           15 * np.exp(-0.5 * ((hours - 18) / 3.0)**2)
price_da = np.maximum(price_da, 10.0)
price_imb_surplus = price_da * 0.7
price_imb_deficit = price_da * 1.3

# Visualize scenario fan
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(N_sites):
    for s in range(S):
        axes[i].plot(hours, pv_scenarios[s][i], 'gold', alpha=0.3, linewidth=0.8)
        axes[i].plot(hours, load_scenarios[s][i], 'red', alpha=0.3, linewidth=0.8)
    axes[i].plot(hours, pv_forecast[i], 'orange', linewidth=2, label='PV forecast')
    axes[i].plot(hours, load_forecast[i], 'darkred', linewidth=2, label='Load forecast')
    axes[i].set_title(f'Site {i}: Scenarios')
    axes[i].set_xlabel('Hour'); axes[i].legend(fontsize=7)

plt.tight_layout()
plt.savefig('/tmp/nb3_scenarios.png', dpi=100)
plt.show()
print(f"Generated {S} scenarios for {N_sites} sites.")


## 2. Two-Stage Stochastic MILP

### Structure

**Stage 1 (Day-Ahead):**
- Decision: nomination $n_t$ for each hour $t$
- These are **here-and-now** decisions, made before uncertainty is revealed

**Stage 2 (Real-Time, per scenario $s$):**
- Decisions: $p^{ch}_{i,t,s}$, $p^{dis}_{i,t,s}$, $p^{buy}_{t,s}$, $p^{sell}_{t,s}$
- These are **recourse** decisions, adapted to each scenario

### Objective
$$\min \sum_{t} \pi^{DA}_t \cdot n_t + \sum_{s} \omega_s \sum_{t} \left[ \pi^{def}_t \cdot \delta^{-}_{t,s} - \pi^{sur}_t \cdot \delta^{+}_{t,s} \right]$$

where $\omega_s$ is the probability of scenario $s$, $\delta^+$ is surplus and $\delta^-$ is deficit.


In [ ]:
prob = pulp.LpProblem("TwoStage_Stochastic_BRP", pulp.LpMinimize)

dt = 1.0
eta_ch = 0.95
eta_dis = 0.95

# Stage 1: Day-ahead nomination (scenario-independent)
nomination = [pulp.LpVariable(f"nom_{t}", lowBound=None) for t in range(T)]

# Stage 2: Per-scenario recourse variables
p_ch = {}    # [i, t, s]
p_dis = {}
soc_var = {}
u_bin = {}
port_buy = {}   # [t, s]
port_sell = {}
surplus_var = {}
deficit_var = {}

for s in range(S):
    for t in range(T):
        port_buy[t,s] = pulp.LpVariable(f"pbuy_{t}_{s}", 0)
        port_sell[t,s] = pulp.LpVariable(f"psell_{t}_{s}", 0)
        surplus_var[t,s] = pulp.LpVariable(f"sur_{t}_{s}", 0)
        deficit_var[t,s] = pulp.LpVariable(f"def_{t}_{s}", 0)

        for i in range(N_sites):
            _, _, E_max_i, P_max_i = site_params[i]
            E_min_i = E_max_i * 0.1
            p_ch[i,t,s] = pulp.LpVariable(f"ch_{i}_{t}_{s}", 0, P_max_i)
            p_dis[i,t,s] = pulp.LpVariable(f"dis_{i}_{t}_{s}", 0, P_max_i)
            soc_var[i,t,s] = pulp.LpVariable(f"soc_{i}_{t}_{s}", E_min_i, E_max_i)
            u_bin[i,t,s] = pulp.LpVariable(f"u_{i}_{t}_{s}", cat='Binary')

# Objective
prob += (
    # Stage 1: DA cost
    pulp.lpSum([(price_da[t]/1000) * nomination[t] * dt for t in range(T)]) +
    # Stage 2: Expected imbalance cost
    pulp.lpSum([
        scenario_probs[s] * (
            (price_imb_deficit[t]/1000) * deficit_var[t,s] * dt -
            (price_imb_surplus[t]/1000) * surplus_var[t,s] * dt
        )
        for s in range(S) for t in range(T)
    ])
), "Expected_Total_Cost"

# Constraints for each scenario
for s in range(S):
    for t in range(T):
        for i in range(N_sites):
            _, _, E_max_i, P_max_i = site_params[i]
            SoC_init_i = E_max_i * 0.5

            # SoC dynamics
            soc_prev = SoC_init_i if t == 0 else soc_var[i, t-1, s]
            prob += (soc_var[i,t,s] == soc_prev +
                     eta_ch * p_ch[i,t,s] * dt - p_dis[i,t,s] * dt / eta_dis,
                     f"SoC_{i}_{t}_{s}")

            # Mutual exclusion
            prob += p_ch[i,t,s] <= P_max_i * u_bin[i,t,s], f"ChMax_{i}_{t}_{s}"
            prob += p_dis[i,t,s] <= P_max_i * (1 - u_bin[i,t,s]), f"DisMax_{i}_{t}_{s}"

        # Portfolio balance for scenario s
        prob += (pulp.lpSum([
            load_scenarios[s][i][t] + p_ch[i,t,s] -
            pv_scenarios[s][i][t] - p_dis[i,t,s]
            for i in range(N_sites)])
            == port_buy[t,s] - port_sell[t,s],
            f"PortBal_{t}_{s}")

        # Imbalance: nomination - actual = surplus - deficit
        actual_net = port_buy[t,s] - port_sell[t,s]
        prob += (nomination[t] - actual_net == surplus_var[t,s] - deficit_var[t,s],
                 f"Imbal_{t}_{s}")

print(f"Problem size: {len(prob.variables())} variables, {len(prob.constraints)} constraints")
print("Solving...")
prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=120))
print(f"Status: {pulp.LpStatus[prob.status]}")
print(f"Expected total cost: {pulp.value(prob.objective):.2f} EUR")


## 3. Results Analysis


In [ ]:
# Extract nominations
nom_vals = [pulp.value(nomination[t]) for t in range(T)]

# Extract per-scenario actuals
actuals = np.zeros((S, T))
surpluses = np.zeros((S, T))
deficits = np.zeros((S, T))
for s in range(S):
    for t in range(T):
        actuals[s, t] = pulp.value(port_buy[t,s]) - pulp.value(port_sell[t,s])
        surpluses[s, t] = pulp.value(surplus_var[t,s])
        deficits[s, t] = pulp.value(deficit_var[t,s])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Nomination vs scenario actuals
axes[0, 0].plot(hours, nom_vals, 'b-o', markersize=4, linewidth=2, label='Nomination', zorder=5)
for s in range(S):
    axes[0, 0].plot(hours, actuals[s], 'gray', alpha=0.3, linewidth=0.8)
axes[0, 0].plot(hours, np.mean(actuals, axis=0), 'r--', linewidth=1.5, label='Mean actual')
axes[0, 0].set_xlabel('Hour'); axes[0, 0].set_ylabel('kW')
axes[0, 0].set_title('Nomination vs Scenario Actuals')
axes[0, 0].legend(); axes[0, 0].axhline(0, color='black', linewidth=0.5)

# Imbalance distribution
mean_surplus = np.mean(surpluses, axis=0)
mean_deficit = np.mean(deficits, axis=0)
axes[0, 1].bar(hours, mean_surplus, color='teal', alpha=0.7, label='E[surplus]')
axes[0, 1].bar(hours, [-v for v in mean_deficit], color='coral', alpha=0.7, label='E[deficit]')
axes[0, 1].set_xlabel('Hour'); axes[0, 1].set_ylabel('kW')
axes[0, 1].set_title('Expected Imbalance'); axes[0, 1].legend()

# Cost breakdown
da_cost = sum((price_da[t]/1000) * nom_vals[t] for t in range(T))
exp_imb_cost = sum(
    scenario_probs[s] * sum(
        (price_imb_deficit[t]/1000) * deficits[s,t] -
        (price_imb_surplus[t]/1000) * surpluses[s,t]
        for t in range(T))
    for s in range(S))

axes[1, 0].bar(['DA Cost', 'E[Imbalance Cost]', 'Total'],
               [da_cost, exp_imb_cost, da_cost + exp_imb_cost],
               color=['steelblue', 'coral', 'purple'])
axes[1, 0].set_ylabel('EUR'); axes[1, 0].set_title('Cost Breakdown')

# SoC fan for site 0
for s in range(S):
    soc_0 = [pulp.value(soc_var[0,t,s]) for t in range(T)]
    axes[1, 1].plot(hours, soc_0, 'steelblue', alpha=0.3, linewidth=0.8)
axes[1, 1].set_xlabel('Hour'); axes[1, 1].set_ylabel('kWh')
axes[1, 1].set_title('Site 0: SoC Across Scenarios')

plt.tight_layout()
plt.savefig('/tmp/nb3_results.png', dpi=100)
plt.show()

print(f"\nDA Cost:              {da_cost:.2f} EUR")
print(f"Expected Imb. Cost:   {exp_imb_cost:.2f} EUR")
print(f"Total Expected Cost:  {da_cost + exp_imb_cost:.2f} EUR")


## 4. Value of Stochastic Solution (VSS)

The **Value of the Stochastic Solution** measures how much we gain by solving the stochastic program instead of using the deterministic solution (based on expected forecasts only).


In [ ]:
# Deterministic solution: solve with mean forecasts only
prob_det = pulp.LpProblem("Deterministic_BRP", pulp.LpMinimize)

nom_det = [pulp.LpVariable(f"dnom_{t}", lowBound=None) for t in range(T)]
pch_det = {}; pdis_det = {}; soc_det = {}; u_det = {}
pbuy_det = [pulp.LpVariable(f"dpbuy_{t}", 0) for t in range(T)]
psell_det = [pulp.LpVariable(f"dpsell_{t}", 0) for t in range(T)]
sur_det = [pulp.LpVariable(f"dsur_{t}", 0) for t in range(T)]
def_det = [pulp.LpVariable(f"ddef_{t}", 0) for t in range(T)]

for i in range(N_sites):
    _, _, E_max_i, P_max_i = site_params[i]
    E_min_i = E_max_i * 0.1
    for t in range(T):
        pch_det[i,t] = pulp.LpVariable(f"dch_{i}_{t}", 0, P_max_i)
        pdis_det[i,t] = pulp.LpVariable(f"ddis_{i}_{t}", 0, P_max_i)
        soc_det[i,t] = pulp.LpVariable(f"dsoc_{i}_{t}", E_min_i, E_max_i)
        u_det[i,t] = pulp.LpVariable(f"du_{i}_{t}", cat='Binary')

prob_det += pulp.lpSum([
    (price_da[t]/1000) * nom_det[t] * dt +
    (price_imb_deficit[t]/1000) * def_det[t] * dt -
    (price_imb_surplus[t]/1000) * sur_det[t] * dt
    for t in range(T)])

for t in range(T):
    for i in range(N_sites):
        _, _, E_max_i, P_max_i = site_params[i]
        SoC_init_i = E_max_i * 0.5
        soc_prev = SoC_init_i if t == 0 else soc_det[i,t-1]
        prob_det += soc_det[i,t] == soc_prev + eta_ch*pch_det[i,t]*dt - pdis_det[i,t]*dt/eta_dis
        prob_det += pch_det[i,t] <= P_max_i * u_det[i,t]
        prob_det += pdis_det[i,t] <= P_max_i * (1 - u_det[i,t])

    # Use mean forecasts
    prob_det += (pulp.lpSum([
        load_forecast[i][t] + pch_det[i,t] - pv_forecast[i][t] - pdis_det[i,t]
        for i in range(N_sites)])
        == pbuy_det[t] - psell_det[t])

    actual_det = pbuy_det[t] - psell_det[t]
    prob_det += nom_det[t] - actual_det == sur_det[t] - def_det[t]

prob_det.solve(pulp.PULP_CBC_CMD(msg=0))
det_cost = pulp.value(prob_det.objective)
det_nom = [pulp.value(nom_det[t]) for t in range(T)]

# Evaluate deterministic nomination under all scenarios
det_costs_per_scenario = []
for s in range(S):
    # Fix nomination to deterministic values, re-solve for each scenario
    prob_eval = pulp.LpProblem(f"Eval_s{s}", pulp.LpMinimize)
    ev_ch = {}; ev_dis = {}; ev_soc = {}; ev_u = {}
    ev_buy = [pulp.LpVariable(f"eb_{t}", 0) for t in range(T)]
    ev_sell = [pulp.LpVariable(f"es_{t}", 0) for t in range(T)]
    ev_sur = [pulp.LpVariable(f"esur_{t}", 0) for t in range(T)]
    ev_def = [pulp.LpVariable(f"edef_{t}", 0) for t in range(T)]

    for i in range(N_sites):
        _, _, E_max_i, P_max_i = site_params[i]
        E_min_i = E_max_i * 0.1
        for t in range(T):
            ev_ch[i,t] = pulp.LpVariable(f"ec_{i}_{t}", 0, P_max_i)
            ev_dis[i,t] = pulp.LpVariable(f"ed_{i}_{t}", 0, P_max_i)
            ev_soc[i,t] = pulp.LpVariable(f"es_{i}_{t}", E_min_i, E_max_i)
            ev_u[i,t] = pulp.LpVariable(f"eu_{i}_{t}", cat='Binary')

    prob_eval += pulp.lpSum([
        (price_da[t]/1000) * det_nom[t] * dt +
        (price_imb_deficit[t]/1000) * ev_def[t] * dt -
        (price_imb_surplus[t]/1000) * ev_sur[t] * dt
        for t in range(T)])

    for t in range(T):
        for i in range(N_sites):
            _, _, E_max_i, P_max_i = site_params[i]
            SoC_init_i = E_max_i * 0.5
            soc_prev = SoC_init_i if t == 0 else ev_soc[i,t-1]
            prob_eval += ev_soc[i,t] == soc_prev + eta_ch*ev_ch[i,t]*dt - ev_dis[i,t]*dt/eta_dis
            prob_eval += ev_ch[i,t] <= P_max_i * ev_u[i,t]
            prob_eval += ev_dis[i,t] <= P_max_i * (1 - ev_u[i,t])

        prob_eval += (pulp.lpSum([
            load_scenarios[s][i][t] + ev_ch[i,t] - pv_scenarios[s][i][t] - ev_dis[i,t]
            for i in range(N_sites)])
            == ev_buy[t] - ev_sell[t])

        actual_ev = ev_buy[t] - ev_sell[t]
        prob_eval += det_nom[t] - actual_ev == ev_sur[t] - ev_def[t]

    prob_eval.solve(pulp.PULP_CBC_CMD(msg=0))
    det_costs_per_scenario.append(pulp.value(prob_eval.objective))

EEV = sum(scenario_probs[s] * det_costs_per_scenario[s] for s in range(S))
stoch_cost = pulp.value(prob.objective)
VSS = EEV - stoch_cost

print(f"Deterministic solution cost (mean forecast):  {det_cost:.2f} EUR")
print(f"Expected cost of using det. nomination (EEV): {EEV:.2f} EUR")
print(f"Stochastic solution cost (SP):                {stoch_cost:.2f} EUR")
print(f"Value of Stochastic Solution (VSS = EEV-SP):  {VSS:.2f} EUR")
if abs(EEV) > 1e-6:
    print(f"VSS as % of EEV:                              {100*VSS/abs(EEV):.1f}%")


## 5. Key Insights for the Thesis

1. **Two-stage stochastic programming** is the natural framework for BRP optimization under forecast uncertainty. The first stage (nomination) is committed before uncertainty resolves; the second stage (BESS dispatch) adapts to the realization.

2. **The VSS quantifies the value of incorporating uncertainty**: even moderate PV/load forecast errors create significant imbalance costs that can be mitigated by hedging via stochastic optimization.

3. **Scenario generation is critical**: the quality of the stochastic solution depends heavily on the scenario set. In practice, scenarios should be derived from ensemble weather forecasts or historical error distributions with proper calibration.

4. **Scalability challenge**: the stochastic MILP has $S$ times more variables than the deterministic model. With many sites and scenarios, decomposition methods (Benders, progressive hedging) become necessary.

5. **Connection to the Czech market**: OTE provides day-ahead prices 12-36 hours ahead, giving time to compute stochastic nominations. Forecast uncertainty is especially relevant for PV-heavy portfolios where cloud cover creates large hour-to-hour variability.

6. **Extension to intraday**: Boomsma et al. (2014) show that adding an intraday trading stage between day-ahead and real-time further reduces expected imbalance costs. This is particularly relevant as CZ is moving toward continuous intraday trading.

---

*Next: Notebook 4 explores decentralized approaches (ADMM) where individual sites retain some autonomy.*
